# 04 · Modelo Preditivo de NPS


**Objetivo:** Construir um modelo capaz de **prever o NPS antes da aplicação da pesquisa**, utilizando apenas dados operacionais disponíveis durante ou logo após a jornada de compra.

---

## Estratégia Adotada

### Por que classificação e não regressão?

Embora o `nps_score` seja uma variável contínua (0 a 10), do ponto de vista do negócio o que realmente importa é saber **se um cliente será Detrator, Neutro ou Promotor**. Isso porquê:

- As ações de negócio são distintas por categoria (reter Detratores, engajar Promotores);
- A classificação é mais robusta a outliers;
- Métricas como F1-Score e Recall são mais acionáveis para times de operação.

A abordagem escolhida é um **modelo de classificação multiclasse** (3 classes: Detrator / Neutro / Promotor).

### Modelo escolhido: Random Forest + XGBoost

Usaremos dois modelos para comparação:
1. **Random Forest** — robusto, interpretável via feature importance, sem necessidade de normalização;
2. **XGBoost** — alta performance, lida bem com dados tabulares e classes desbalanceadas.

---


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from src.utils import load_raw, add_nps_category, remove_leakage_cols, set_style, save_fig, PALETTE

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score, roc_auc_score
)
from sklearn.pipeline import Pipeline

# XGBoost
from xgboost import XGBClassifier

set_style()
print('Ambiente pronto.')

## 1. Carregamento e Preparação dos Dados

In [ ]:
df = load_raw()
df = add_nps_category(df)

# ── Definição da Target ──────────────────────────────────────────────────────
# Usamos nps_category (Detrator / Neutro / Promotor) como alvo de classificação.
# Isso resolve o problema de negócio: prever o PERFIL do cliente, não a nota exata.
TARGET_COL = 'nps_category'

# ── Remoção de Data Leakage ──────────────────────────────────────────────────
df_features = remove_leakage_cols(df)
df_features = df_features.drop(columns=['customer_id', 'order_id'], errors='ignore')

# ── Encoding de variáveis categóricas ───────────────────────────────────────
df_features = pd.get_dummies(df_features, columns=['customer_region'], drop_first=False)

# ── Feature Engineering ──────────────────────────────────────────────────────
df_features['discount_ratio'] = df_features['discount_value'] / (df_features['order_value'] + 1e-9)
df_features['has_delay']      = (df_features['delivery_delay_days'] > 0).astype(int)
df_features['high_contact']   = (df_features['customer_service_contacts'] >= 3).astype(int)
df_features['freight_per_item'] = df_features['freight_value'] / (df_features['items_quantity'] + 1e-9)

# ── Separação X e y ──────────────────────────────────────────────────────────
X = df_features.drop(columns=[TARGET_COL], errors='ignore')
y_raw = df[TARGET_COL]

# Encode target como inteiro (necessário para XGBoost)
le = LabelEncoder()
y = le.fit_transform(y_raw)  # Detrator=0, Neutro=1, Promotor=2
class_names = le.classes_

print('Classes:', class_names)
print('Distribuição:', pd.Series(y_raw).value_counts().to_dict())
print('\nFeatures utilizadas:', X.shape[1])
print(X.columns.tolist())

## 2. Divisão Treino / Teste

Usamos **Stratified Split** para garantir que a proporção de classes seja mantida tanto no treino quanto no teste — fundamental quando há desbalanceamento entre Detratores, Neutros e Promotores.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,       # 80% treino, 20% teste
    random_state=42,
    stratify=y            # mantém proporção das classes
)

print(f'Treino: {X_train.shape[0]:,} amostras')
print(f'Teste:  {X_test.shape[0]:,} amostras')
print('\nDistribuição no treino:')
print(pd.Series(le.inverse_transform(y_train)).value_counts())

## 3. Treinamento dos Modelos

In [ ]:
# ── Random Forest ────────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',   # lida com desbalanceamento
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print('Random Forest treinado.')

# ── XGBoost ──────────────────────────────────────────────────────────────────
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train)
print('XGBoost treinado.')

## 4. Avaliação dos Modelos

In [ ]:
def avaliar_modelo(modelo, nome, X_test, y_test, class_names):
    """Avalia um modelo e exibe métricas + matriz de confusão."""
    y_pred = modelo.predict(X_test)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')

    print(f'\n{'='*55}')
    print(f'  {nome}')
    print(f'{'='*55}')
    print(f'  F1-Score Macro:    {f1_macro:.4f}')
    print(f'  F1-Score Weighted: {f1_weighted:.4f}')
    print(f'\n{classification_report(y_test, y_pred, target_names=class_names)}')

    # Matriz de confusão
    fig, ax = plt.subplots(figsize=(7, 5))
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Matriz de Confusão — {nome}')
    save_fig(fig, f'confusion_matrix_{nome.lower().replace(" ", "_")}.png')
    plt.show()

    return y_pred, f1_macro


_, f1_rf = avaliar_modelo(rf, 'Random Forest', X_test, y_test, class_names)
_, f1_xgb = avaliar_modelo(xgb, 'XGBoost', X_test, y_test, class_names)

## 5. Validação Cruzada (Cross-Validation)

In [ ]:
# Validação cruzada estratificada com 5 folds
# Garante que os resultados não dependem de uma divisão aleatória específica

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_rf = cross_val_score(rf, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)
cv_xgb = cross_val_score(xgb, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)

print('Random Forest — CV F1-Macro:')
print(f'  Média: {cv_rf.mean():.4f} | Desvio: {cv_rf.std():.4f}')
print(f'  Por fold: {cv_rf.round(4)}')

print('\nXGBoost — CV F1-Macro:')
print(f'  Média: {cv_xgb.mean():.4f} | Desvio: {cv_xgb.std():.4f}')
print(f'  Por fold: {cv_xgb.round(4)}')

## 6. Importância das Features

In [ ]:
# Melhor modelo para feature importance
melhor_modelo = xgb if f1_xgb >= f1_rf else rf
nome_melhor = 'XGBoost' if f1_xgb >= f1_rf else 'Random Forest'
print(f'Melhor modelo: {nome_melhor}')

importances = pd.Series(
    melhor_modelo.feature_importances_,
    index=X.columns
).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
importances.plot(kind='barh', ax=ax, color='#3498DB')
ax.invert_yaxis()
ax.set_title(f'Top 15 Features Mais Importantes — {nome_melhor}')
ax.set_xlabel('Importância')
ax.axvline(0, color='black', linewidth=0.8)
save_fig(fig, 'feature_importance.png')
plt.show()

print('\nTop 10 features:')
print(importances.head(10))

## 7. Salvando o Modelo

In [ ]:
import joblib
from pathlib import Path

MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Salva modelo e encoder
joblib.dump(melhor_modelo, MODELS_DIR / 'nps_classifier.pkl')
joblib.dump(le, MODELS_DIR / 'label_encoder.pkl')
joblib.dump(X.columns.tolist(), MODELS_DIR / 'feature_names.pkl')

print(f'Modelo salvo em: {MODELS_DIR / "nps_classifier.pkl"}')
print(f'Label encoder salvo em: {MODELS_DIR / "label_encoder.pkl"}')

## 8. Como o Modelo Seria Utilizado na Prática

```python
import joblib
import pandas as pd

# Carrega modelo e encoder
model     = joblib.load('models/nps_classifier.pkl')
encoder   = joblib.load('models/label_encoder.pkl')
features  = joblib.load('models/feature_names.pkl')

# Novo pedido (dados disponíveis ANTES da pesquisa de NPS)
novo_pedido = pd.DataFrame([{
    'customer_age': 35,
    'customer_tenure_months': 24,
    'order_value': 350.0,
    'items_quantity': 2,
    'discount_value': 30.0,
    'payment_installments': 3,
    'delivery_time_days': 7,
    'delivery_delay_days': 5,  # ← atraso crítico!
    'freight_value': 25.0,
    'delivery_attempts': 1,
    'customer_service_contacts': 2,
    'resolution_time_days': 3,
    'complaints_count': 1,
    'customer_region_Sudeste': 1,
    # ... demais features
}])

# Predição
pred_encoded = model.predict(novo_pedido[features])
pred_classe  = encoder.inverse_transform(pred_encoded)
print(f'Perfil previsto: {pred_classe[0]}')  # → 'Detrator'
# → Aciona alerta para equipe de atendimento intervir proativamente
```

---

## 9. Limitações e Cuidados

| Risco | Descrição |
|---|---|
| **Data Leakage** | `csat_internal_score` e `repeat_purchase_30d` foram excluídos — não estariam disponíveis em produção |
| **Drift temporal** | O modelo deve ser re-treinado periodicamente à medida que o comportamento do cliente muda |
| **Classe Neutro** | Clientes Neutros são os mais difíceis de classificar; monitorar seu Recall |
| **Causalidade** | Alta importância de uma feature não significa que agir nela causará mudança no NPS — correlação ≠ causalidade |
| **Generalização** | O modelo foi treinado com 2.500 registros; validar com dados mais recentes antes de produção |
